# Task 3 — Metric Calculations

Compute network centrality metrics (degree, betweenness, clustering) and analyze shortest paths for drug repurposing. Identify hub nodes (highly connected) and bottleneck nodes (critical connectors) with biological interpretation.

## 1. Initialize Project Environment

In [1]:
"""Setup and imports for Lab 9 Task 3 - Metric Calculations."""
import logging
import sys
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Dict, List, Tuple

import networkx as nx
import pandas as pd

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S",
)

print(f"Python {sys.version}")
print(f"networkx {nx.__version__}")

Python 3.12.3 (main, Jan  8 2026, 11:30:50) [GCC 13.3.0]
networkx 3.3


## 2. Define Configuration Parameters

In [2]:
@dataclass
class Task3Config:
    """Configuration for network metric calculations."""

    handle: str = "AndreiCod"
    export_dir: Path = Path("./artifacts")
    network_file: str = "task2_network.gml"

    def __post_init__(self):
        self.export_dir.mkdir(parents=True, exist_ok=True)

    def describe(self) -> Dict[str, str]:
        info = asdict(self)
        info["export_dir"] = str(info["export_dir"])
        return info


CONFIG = Task3Config()
CONFIG.describe()

{'handle': 'AndreiCod',
 'export_dir': 'artifacts',
 'network_file': 'task2_network.gml'}

## 3. Implement Core Functionality

In [3]:
def load_network(export_dir: Path, filename: str) -> nx.Graph:
    """Load network from Task 2."""
    filepath = export_dir / filename
    if not filepath.exists():
        raise FileNotFoundError(f"Network file not found: {filepath}. Run Task2 first.")
    G = nx.read_gml(filepath)
    logging.info(
        "Loaded network: %d nodes, %d edges", G.number_of_nodes(), G.number_of_edges()
    )
    return G


# Load network
G = load_network(CONFIG.export_dir, CONFIG.network_file)
print(
    f"[OK] Loaded network with {G.number_of_nodes()} nodes and {G.number_of_edges()} edges"
)

22:31:02 | INFO | Loaded network: 26616 nodes, 38150 edges


[OK] Loaded network with 26616 nodes and 38150 edges


In [4]:
def compute_centrality_metrics(G: nx.Graph) -> pd.DataFrame:
    """
    Compute degree, betweenness centrality, and clustering coefficient.
    For large networks, use approximation for betweenness.
    """
    logging.info("Computing degree centrality...")
    degree_cent = nx.degree_centrality(G)

    logging.info("Computing betweenness centrality (using k=500 sample for speed)...")
    # Use sampling for large networks
    k = min(500, G.number_of_nodes())
    betweenness_cent = nx.betweenness_centrality(G, k=k)

    logging.info("Computing clustering coefficient...")
    clustering_coef = nx.clustering(G)

    # Get node attributes
    node_types = nx.get_node_attributes(G, "node_type")

    # Build dataframe
    metrics_data = []
    for node in G.nodes():
        metrics_data.append(
            {
                "node": node,
                "node_type": node_types.get(node, "Unknown"),
                "degree": G.degree(node),
                "degree_centrality": degree_cent[node],
                "betweenness_centrality": betweenness_cent[node],
                "clustering_coefficient": clustering_coef[node],
            }
        )

    df = pd.DataFrame(metrics_data)
    df = df.sort_values("degree_centrality", ascending=False)

    logging.info("Computed centrality metrics for %d nodes", len(df))
    return df


# Compute metrics
metrics_df = compute_centrality_metrics(G)
print(f"[OK] Computed metrics for {len(metrics_df)} nodes")
metrics_df.head(10)

22:31:02 | INFO | Computing degree centrality...


22:31:02 | INFO | Computing betweenness centrality (using k=500 sample for speed)...
22:31:26 | INFO | Computing clustering coefficient...
22:31:27 | INFO | Computed centrality metrics for 26616 nodes


[OK] Computed metrics for 26616 nodes


,node,node_type,degree,degree_centrality,betweenness_centrality,clustering_coefficient
10045,Fostamatinib,Drug,298,0.011197,0.037852,0.0
20200,Gene:ADRA1A,Gene,176,0.006613,0.002501,0.0
19936,Gene:PTGS2,Gene,147,0.005523,0.005573,0.0
20217,Gene:CHRM1,Gene,147,0.005523,0.002184,0.0
147,NADH,Drug,146,0.005486,0.014973,0.0
22484,"Syndrome, Alzheimer'S Disease",Disease,145,0.005448,0.005462,0.0
8149,Copper,Drug,145,0.005448,0.017160,0.0
22483,"Parkinson'S Disease, Chronic",Disease,145,0.005448,0.005462,0.0
18241,Cyclin-dependent kinase 2,Target,137,0.005147,0.004682,0.0
24109,Solid Tumors,Disease,136,0.005110,0.015058,0.0


## 4. Validate with Unit Tests

In [5]:
# Validate metrics
assert len(metrics_df) == G.number_of_nodes(), "Mismatch in node count"
assert "degree_centrality" in metrics_df.columns, "Missing degree_centrality"
assert "betweenness_centrality" in metrics_df.columns, "Missing betweenness_centrality"
assert "clustering_coefficient" in metrics_df.columns, "Missing clustering_coefficient"
assert (metrics_df["degree_centrality"] >= 0).all(), "Invalid degree centrality values"
assert (metrics_df["degree_centrality"] <= 1).all(), "Degree centrality out of range"

print("[OK] All metric validations passed")

# Display top hubs by node type
print("\n=== Top 10 Hubs by Degree Centrality ===")
print(metrics_df.head(10)[["node", "node_type", "degree", "degree_centrality"]])

print("\n=== Top 10 Bottleneck Nodes by Betweenness ===")
top_betweenness = metrics_df.nlargest(10, "betweenness_centrality")
print(top_betweenness[["node", "node_type", "betweenness_centrality"]])

print("\n=== Hub Summary by Node Type ===")
for node_type in ["Drug", "Target", "Disease"]:
    type_df = metrics_df[metrics_df["node_type"] == node_type]
    top = type_df.head(3)
    print(f"\nTop 3 {node_type}s:")
    for _, row in top.iterrows():
        print(
            f"  {row['node'][:40]}: degree={row['degree']}, betweenness={row['betweenness_centrality']:.4f}"
        )

[OK] All metric validations passed

=== Top 10 Hubs by Degree Centrality ===
                                node node_type  degree  degree_centrality
10045                   Fostamatinib      Drug     298           0.011197
20200                    Gene:ADRA1A      Gene     176           0.006613
19936                     Gene:PTGS2      Gene     147           0.005523
20217                     Gene:CHRM1      Gene     147           0.005523
147                             NADH      Drug     146           0.005486
22484  Syndrome, Alzheimer'S Disease   Disease     145           0.005448
8149                          Copper      Drug     145           0.005448
22483   Parkinson'S Disease, Chronic   Disease     145           0.005448
18241      Cyclin-dependent kinase 2    Target     137           0.005147
24109                   Solid Tumors   Disease     136           0.005110

=== Top 10 Bottleneck Nodes by Betweenness ===
                                                node node_typ

In [6]:
def compute_drug_disease_paths(G: nx.Graph, max_pairs: int = 5000) -> pd.DataFrame:
    """
    Compute shortest paths between drugs and diseases.
    KEY for drug repurposing: paths Drug → Target → Gene → Disease suggest repurposing candidates.

    The assignment specifies looking for:
    - Which drugs are closest to disease-associated genes?
    - Can this suggest repurposing candidates?
    """
    drug_nodes = [n for n, d in G.nodes(data=True) if d.get("node_type") == "Drug"]
    disease_nodes = [
        n for n, d in G.nodes(data=True) if d.get("node_type") == "Disease"
    ]

    logging.info(
        "Computing shortest paths: %d drugs × %d diseases",
        len(drug_nodes),
        len(disease_nodes),
    )

    paths_data = []
    count = 0

    # Process all drug-disease pairs, prioritizing shorter paths
    for drug in drug_nodes:
        if count >= max_pairs:
            break

        for disease in disease_nodes:
            if count >= max_pairs:
                break
            try:
                path_length = nx.shortest_path_length(G, drug, disease)

                # Focus on indirect connections (path > 1) that could indicate repurposing
                if path_length > 1 and path_length <= 5:  # Reasonable path lengths
                    path = nx.shortest_path(G, drug, disease)

                    # Determine intermediate node types
                    intermediate_types = []
                    for node in path[1:-1]:
                        ntype = G.nodes[node].get("node_type", "Unknown")
                        intermediate_types.append(ntype)

                    # Use full node names (increased from 30 to 50 chars) for better readability
                    path_str = " → ".join([str(p)[:50] for p in path])

                    paths_data.append(
                        {
                            "drug": drug,
                            "disease": disease,
                            "path_length": path_length,
                            "path": path_str,
                            "path_full": " → ".join(
                                [str(p) for p in path]
                            ),  # Full path for analysis
                            "intermediate_nodes": path_length - 1,
                            "intermediate_types": " → ".join(intermediate_types),
                            "has_gene_intermediate": "Gene" in intermediate_types,
                            "has_target_intermediate": "Target" in intermediate_types,
                        }
                    )
                    count += 1

            except nx.NetworkXNoPath:
                pass

    df = pd.DataFrame(paths_data)
    if len(df) > 0:
        # Sort by path length (shorter = more interesting for repurposing)
        df = df.sort_values("path_length")

    logging.info("Found %d drug-disease paths (max %d)", len(df), max_pairs)
    return df


# Compute drug-disease shortest paths
paths_df = compute_drug_disease_paths(G, max_pairs=5000)
print(f"[OK] Computed {len(paths_df)} drug-disease paths")

# Show distribution of path lengths
if len(paths_df) > 0:
    print("\n=== Path Length Distribution ===")
    print(paths_df["path_length"].value_counts().sort_index())

    print("\n=== Sample Shortest Paths (potential repurposing candidates) ===")
    display(paths_df.head(15))

22:31:27 | INFO | Computing shortest paths: 16575 drugs × 4360 diseases
22:31:28 | INFO | Found 5000 drug-disease paths (max 5000)


[OK] Computed 5000 drug-disease paths

=== Path Length Distribution ===
path_length
3     502
4      18
5    4480
Name: count, dtype: int64

=== Sample Shortest Paths (potential repurposing candidates) ===


,drug,disease,path_length,path,path_full,intermediate_nodes,intermediate_types,has_gene_intermediate,has_target_intermediate
2910,Cetuximab,Colorectal Carcinoma That,3,Cetuximab → Epidermal growth factor receptor →...,Cetuximab → Epidermal growth factor receptor →...,2,Target → Drug,False,True
361,Lepirudin,Discrete Aldosterone-Producing Adrenal Adenomas,3,Lepirudin → Acute Myocardial Infarction → Gene...,Lepirudin → Acute Myocardial Infarction → Gene...,2,Disease → Gene,True,False
360,Lepirudin,Primary Hyperaldosteronism,3,Lepirudin → Acute Myocardial Infarction → Gene...,Lepirudin → Acute Myocardial Infarction → Gene...,2,Disease → Gene,True,False
359,Lepirudin,"Hypertension, As Add-On",3,Lepirudin → Acute Myocardial Infarction → Gene...,Lepirudin → Acute Myocardial Infarction → Gene...,2,Disease → Gene,True,False
358,Lepirudin,Bilateral Micro,3,Lepirudin → Acute Myocardial Infarction → Gene...,Lepirudin → Acute Myocardial Infarction → Gene...,2,Disease → Gene,True,False
2971,Cetuximab,Zinc Deficiency/Its Consequences,3,Cetuximab → Complement C1q subcomponent subuni...,Cetuximab → Complement C1q subcomponent subuni...,2,Target → Drug,False,True
2970,Cetuximab,"Ear Infections, As Well",3,Cetuximab → Complement C1q subcomponent subuni...,Cetuximab → Complement C1q subcomponent subuni...,2,Target → Drug,False,True
3002,Cetuximab,"Cancer, Leukemia (Lymphoid), Lung",3,Cetuximab → Epidermal growth factor receptor →...,Cetuximab → Epidermal growth factor receptor →...,2,Target → Drug,False,True
2898,Cetuximab,Whose Tumors Overexpress,3,Cetuximab → Epidermal growth factor receptor →...,Cetuximab → Epidermal growth factor receptor →...,2,Target → Drug,False,True
2897,Cetuximab,Breast Cancer Whose Tumors,3,Cetuximab → Epidermal growth factor receptor →...,Cetuximab → Epidermal growth factor receptor →...,2,Target → Drug,False,True


## 5. Export Results

In [7]:
# Export metrics
EXPORT_DIR = CONFIG.export_dir

# Centrality metrics
metrics_file = EXPORT_DIR / "task3_centrality_metrics.csv"
metrics_df.to_csv(metrics_file, index=False)

# Shortest paths
paths_file = EXPORT_DIR / "task3_drug_disease_paths.csv"
paths_df.to_csv(paths_file, index=False)

# Export hub genes summary (for Task 4)
hub_targets = metrics_df[metrics_df["node_type"] == "Target"].nlargest(
    20, "degree_centrality"
)
hub_file = EXPORT_DIR / "task3_hub_targets.csv"
hub_targets.to_csv(hub_file, index=False)

print(f"[OK] Centrality metrics saved to: {metrics_file.resolve()}")
print(f"[OK] Drug-disease paths saved to: {paths_file.resolve()}")
print(f"[OK] Hub targets saved to: {hub_file.resolve()}")

[OK] Centrality metrics saved to: /home/rbals/git/daha-bdhb/BDHB-lab/labs/09_repurposing/assignments/artifacts/task3_centrality_metrics.csv
[OK] Drug-disease paths saved to: /home/rbals/git/daha-bdhb/BDHB-lab/labs/09_repurposing/assignments/artifacts/task3_drug_disease_paths.csv
[OK] Hub targets saved to: /home/rbals/git/daha-bdhb/BDHB-lab/labs/09_repurposing/assignments/artifacts/task3_hub_targets.csv
